In [6]:
import pyodbc
import pandas as pd

conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost,1433;"
    "DATABASE=SotexHackathon;"
    "UID=sa;"
    "PWD=SotexSolutions123!;"
    "TrustServerCertificate=yes;"
)

conn = pyodbc.connect(conn_str)

query = """
SELECT 
    TABLE_NAME,
    COLUMN_NAME,
    DATA_TYPE,
    CHARACTER_MAXIMUM_LENGTH,
    IS_NULLABLE,
    CASE WHEN COLUMNPROPERTY(OBJECT_ID(TABLE_NAME), COLUMN_NAME, 'IsIdentity') = 1 THEN 'YES' ELSE 'NO' END as IS_IDENTITY
FROM INFORMATION_SCHEMA.COLUMNS
ORDER BY TABLE_NAME, ORDINAL_POSITION
"""

df = pd.read_sql(query, conn)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 30)

for table in df['TABLE_NAME'].unique():
    print("\n" + "═" * 90)
    print(f" TABELA: {table}")
    print("═" * 90)
    
    table_df = df[df['TABLE_NAME'] == table].copy()
    
    def format_type(row):
        if row['DATA_TYPE'] in ('nvarchar', 'varchar'):
            if pd.notna(row['CHARACTER_MAXIMUM_LENGTH']):
                if row['CHARACTER_MAXIMUM_LENGTH'] == -1:
                    return f"{row['DATA_TYPE']}(MAX)"
                else:
                    return f"{row['DATA_TYPE']}({int(row['CHARACTER_MAXIMUM_LENGTH'])})"
        return row['DATA_TYPE']
    
    table_df['DATA_TYPE'] = table_df.apply(format_type, axis=1)
    
    for _, row in table_df.iterrows():
        nullable = "NULL" if row['IS_NULLABLE'] == 'YES' else "NOT NULL"
        
        print(f"  • {row['COLUMN_NAME']:<25} {row['DATA_TYPE']:<20} {nullable:<10}")
    

conn.close()



══════════════════════════════════════════════════════════════════════════════════════════
 TABELA: Channels
══════════════════════════════════════════════════════════════════════════════════════════
  • Id                        int                  NOT NULL  
  • Name                      nvarchar(100)        NULL      
  • Unit                      nvarchar(30)         NULL      

══════════════════════════════════════════════════════════════════════════════════════════
 TABELA: DistributionSubstation
══════════════════════════════════════════════════════════════════════════════════════════
  • Id                        int                  NOT NULL  
  • Name                      nvarchar(100)        NULL      
  • MeterId                   int                  NULL      
  • Feeder11Id                int                  NULL      
  • Feeder33Id                int                  NULL      
  • NameplateRating           int                  NULL      
  • Latitude              

C:\Users\Tanja\AppData\Local\Temp\ipykernel_23688\2983483474.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from database import engine
from sqlalchemy import text

def run_eda(feeder_id: int, hours: int = 300000):
    print(f"--- Pokrećem EDA za Feeder {feeder_id} ---")
    
    # 1. Izvlačenje podataka (koristimo tvoju logiku)
    query = text("""
        SELECT m.Val AS value, m.Ts AS timestamp, m.Mid AS meter_id
        FROM dbo.MeterReadTfes m
        JOIN dbo.Meters me ON m.Mid = me.Id
        JOIN dbo.DistributionSubstation ds ON ds.MeterId = me.Id
        WHERE ds.Feeder11Id = :feeder_id
        AND m.Ts >= DATEADD(hour, -:hours, (SELECT MAX(Ts) FROM dbo.MeterReadTfes))
        ORDER BY m.Ts
    """)
    
    with engine.connect() as conn:
        df = pd.read_sql(query, conn, params={"feeder_id": feeder_id, "hours": hours})
    
    if df.empty:
        print("Nema podataka za analizu.")
        return

    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # Postavljanje stila za grafikone
    plt.style.use('seaborn-v0_8')
    fig, axes = plt.subplots(3, 1, figsize=(12, 18))

    # --- GRAFIKON 1: Potrošnja kroz vreme (Time Series) ---
    sns.lineplot(ax=axes[0], data=df, x='timestamp', y='value', hue='meter_id')
    axes[0].set_title(f'Potrošnja energije kroz vreme - Feeder {feeder_id}', fontsize=15)
    axes[0].set_ylabel('Vrednost (kWh/snaga)')

    # --- GRAFIKON 2: Detekcija rupa (Histogram gapova) ---
    df['gap'] = df.groupby('meter_id')['timestamp'].diff().dt.total_seconds() / 60
    # Filtriramo normalne gapove (30 min) da vidimo samo anomalije
    gaps_df = df[df['gap'] > 35] 
    
    if not gaps_df.empty:
        sns.histplot(ax=axes[1], x=gaps_df['gap'], bins=30, color='red')
        axes[1].set_title('Distribucija trajanja prekida (u minutima)', fontsize=15)
        axes[1].set_xlabel('Minuti bez podataka')
    else:
        axes[1].text(0.5, 0.5, 'Nema značajnih prekida u podacima', ha='center')

    # --- GRAFIKON 3: Boxplot opterećenja ---
    # Ovo pokazuje da li ima "iskakanja" (outliera) u snazi
    sns.boxplot(ax=axes[2], x='meter_id', y='value', data=df)
    axes[2].set_title('Analiza opterećenja po brojilima (Outliers)', fontsize=15)

    plt.tight_layout()
    plt.savefig('eda_report.png')
    print("--- EDA završena! Grafikon je sačuvan kao 'eda_report.png' ---")
    plt.show()

if __name__ == "__main__":
    # Testiraj za feeder 1
    run_eda(feeder_id=1)

ModuleNotFoundError: No module named 'database'